In [1]:
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix,
    cohen_kappa_score, matthews_corrcoef
)
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

df = pd.read_csv("/kaggle/input/datasets/iqsnguyn/lancuoidibennhau/Denguediseasesdataset1003.csv_clean.csv")

TARGET_COL = 'Final Output'

if df[TARGET_COL].dtype == 'object' or df[TARGET_COL].dtype.name == 'category':
    _target_le = LabelEncoder()
    df[TARGET_COL] = _target_le.fit_transform(df[TARGET_COL])
    print(f"Mã hóa nhãn: {dict(zip(_target_le.classes_, _target_le.transform(_target_le.classes_)))}")

target = df[TARGET_COL].values.astype(int)

print(f"Tổng số mẫu: {len(df)}")
print(f"Kích thước dữ liệu: {df.shape}")
print(f"Phân bố nhãn: Âm tính={np.sum(target==0)} | Dương tính={np.sum(target==1)}")
print(f"Tỷ lệ dương tính: {target.mean()*100:.1f}%")

categorical_cols = ['Sex', 'Differential Count', 'RBC PANEL']

numerical_cols = ['Haemoglobin', 'Age', 'WBC Count', 'Platelet Count', 'PDW' ]

missing_cols = [c for c in categorical_cols + numerical_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Các cột sau không tồn tại trong data: {missing_cols}")

print(f"\nĐặc trưng phân loại ({len(categorical_cols)}): {categorical_cols}")
print(f"Đặc trưng số học    ({len(numerical_cols)}): {numerical_cols}")
print(f"Tổng đặc trưng: {len(categorical_cols) + len(numerical_cols)}")

class TabularPreprocessor:
    def __init__(self):
        self.label_encoders = {}
        self.scaler = StandardScaler()
        self.categorical_dims = []

    def fit(self, df, categorical_cols, numerical_cols):
        self.categorical_dims = []
        self.label_encoders = {}
        for col in categorical_cols:
            le = LabelEncoder()
            df_temp = df[col].fillna('missing').astype(str)
            le.fit(df_temp)
            self.label_encoders[col] = le
            self.categorical_dims.append(len(le.classes_))
        if numerical_cols:
            df_num = df[numerical_cols].fillna(df[numerical_cols].median())
            self.scaler.fit(df_num)
        return self

    def transform(self, df, categorical_cols, numerical_cols):
        df_processed = df.copy()
        for col in categorical_cols:
            df_temp = df_processed[col].fillna('missing').astype(str)
            le = self.label_encoders[col]
            df_processed[col] = df_temp.map(
                lambda x, _le=le: _le.transform([x])[0]
                if x in _le.classes_ else 0
            )
        if numerical_cols:
            df_num = df_processed[numerical_cols].fillna(
                df_processed[numerical_cols].median()
            )
            df_processed[numerical_cols] = self.scaler.transform(df_num)
        return df_processed


# KIẾN TRÚC SAINT

class SAINT(nn.Module):
    def __init__(self, categories, num_continuous, dim=32, depth=6, heads=8,
                 attn_dropout=0.1, ff_dropout=0.1, mlp_hidden_mults=(4, 2)):
        super().__init__()
        self.num_categories = len(categories)
        self.num_continuous = num_continuous

        self.categorical_embeds = nn.ModuleList([
            nn.Embedding(cat_size, dim) for cat_size in categories
        ])

        if num_continuous > 0:
            self.numerical_embed = nn.Linear(num_continuous, dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=dim * 4,
            dropout=attn_dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        total_features = self.num_categories + (1 if num_continuous > 0 else 0)
        assert total_features > 0, "Phải có ít nhất 1 đặc trưng!"

        mlp_layers = []
        input_dim = dim * total_features
        for mult in mlp_hidden_mults:
            out_dim = input_dim // mult
            out_dim = max(out_dim, 1)
            mlp_layers.extend([
                nn.Linear(input_dim, out_dim),
                nn.ReLU(),
                nn.Dropout(ff_dropout)
            ])
            input_dim = out_dim
        mlp_layers.append(nn.Linear(input_dim, 1))
        self.mlp = nn.Sequential(*mlp_layers)

    def forward(self, x_cat, x_num=None):
        embeddings = []
        if self.num_categories > 0:
            for i, embed in enumerate(self.categorical_embeds):
                embeddings.append(embed(x_cat[:, i]))
        if x_num is not None and self.num_continuous > 0:
            embeddings.append(self.numerical_embed(x_num))
        x = torch.stack(embeddings, dim=1)
        x = self.transformer(x)
        x = x.flatten(1)
        return self.mlp(x)

class DengueDataset(Dataset):
    def __init__(self, cat_data, num_data=None, labels=None):
        self.cat_data = torch.LongTensor(cat_data)
        self.num_data = (torch.FloatTensor(num_data)
                         if num_data is not None else torch.FloatTensor([]))
        self.labels = torch.FloatTensor(labels) if labels is not None else None

    def __len__(self):
        return len(self.cat_data)

    def __getitem__(self, idx):
        if self.labels is not None:
            return self.cat_data[idx], self.num_data[idx], self.labels[idx]
        return self.cat_data[idx], self.num_data[idx]


# CLASS-WEIGHTED BCE LOSS
class WeightedBCEWithLogitsLoss(nn.Module):
    """
    Class-weighted Binary Cross Entropy:
        weight_pos = n_total / (2 * n_pos)
        weight_neg = n_total / (2 * n_neg)
    Mỗi sample được nhân trọng số tương ứng class của nó.
    """
    def __init__(self, weight_pos: float, weight_neg: float):
        super().__init__()
        self.weight_pos = weight_pos
        self.weight_neg = weight_neg

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none'
        )
        w_pos = torch.tensor(self.weight_pos, device=logits.device, dtype=logits.dtype)
        w_neg = torch.tensor(self.weight_neg, device=logits.device, dtype=logits.dtype)
        weights = torch.where(targets == 1, w_pos, w_neg)
        return (bce * weights).mean()

def train_saint_model(train_dataset, val_dataset, categorical_dims, num_continuous,
                      epochs=50, batch_size=64, lr=0.001, device='cpu',
                      weight_pos=1.0, weight_neg=1.0):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
    model = SAINT(
        categories=categorical_dims if categorical_dims else [],
        num_continuous=num_continuous,
        dim=32, depth=4, heads=8,
        attn_dropout=0.1, ff_dropout=0.1
    ).to(device)

    criterion = WeightedBCEWithLogitsLoss(weight_pos, weight_neg)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_auc   = 0.0
    best_model_state = None

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for cat_batch, num_batch, y_batch in train_loader:
            cat_batch = cat_batch.to(device)
            num_batch = num_batch.to(device) if num_batch.numel() > 0 else None
            y_batch   = y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(cat_batch, num_batch).squeeze()
            if outputs.dim() == 0:
                outputs = outputs.unsqueeze(0)
            loss = criterion(outputs, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())

        scheduler.step()

        model.eval()
        val_losses, val_preds, val_labels = [], [], []
        with torch.no_grad():
            for cat_batch, num_batch, y_batch in val_loader:
                cat_batch = cat_batch.to(device)
                num_batch = num_batch.to(device) if num_batch.numel() > 0 else None
                y_batch   = y_batch.to(device)

                outputs = model(cat_batch, num_batch).squeeze()
                if outputs.dim() == 0:
                    outputs = outputs.unsqueeze(0)
                val_losses.append(criterion(outputs, y_batch).item())
                val_preds.extend(torch.sigmoid(outputs).cpu().numpy().tolist())
                val_labels.extend(y_batch.cpu().numpy().tolist())

        val_auc = roc_auc_score(val_labels, val_preds)
        if val_auc > best_val_auc:
            best_val_auc     = val_auc
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}

        if epoch % 10 == 0:
            print(f"  Epoch {epoch:3d}: Train Loss={np.mean(train_losses):.4f} | "
                  f"Val Loss={np.mean(val_losses):.4f} | Val AUC={val_auc:.4f}")

    if best_model_state:
        model.load_state_dict(best_model_state)
    return model, best_val_auc


# MCC-BASED THRESHOLD TUNING
def find_best_threshold_mcc(y_true, y_prob, thresholds=None):
    """
    Tìm ngưỡng tối ưu dựa trên MCC.
    Quét từ 0.01 đến 0.99 (bước 0.01).
    """
    if thresholds is None:
        thresholds = np.arange(0.01, 1.00, 0.01)

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    best_mcc, best_threshold = -2.0, 0.5

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_pred)) > 1 else 0.0
        if mcc > best_mcc:
            best_mcc, best_threshold = mcc, t

    return float(best_threshold), float(best_mcc)

METRICS_COLS = ['AUROC', 'Accuracy', 'Sensitivity', 'Specificity',
                'PPV', 'NPV', 'Kappa', 'MCC', 'Threshold']

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (np.array(y_prob) >= threshold).astype(int)
    y_true = np.array(y_true)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        'AUROC':       roc_auc_score(y_true, y_prob),
        'Accuracy':    accuracy_score(y_true, y_pred),
        'Sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0.0,
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'PPV':         tp / (tp + fp) if (tp + fp) > 0 else 0.0,
        'NPV':         tn / (tn + fn) if (tn + fn) > 0 else 0.0,
        'Kappa':       cohen_kappa_score(y_true, y_pred),
        'MCC':         matthews_corrcoef(y_true, y_pred)
                       if len(np.unique(y_pred)) > 1 else 0.0,
        'Threshold':   float(threshold),
        'TP': int(tp), 'FP': int(fp),
        'TN': int(tn), 'FN': int(fn),
    }


def print_metrics_table(m, title=''):
    if title:
        print(f"\n{'='*55}")
        print(f"  {title}")
        print(f"{'='*55}")
    display_cols = ['AUROC', 'Accuracy', 'Sensitivity', 'Specificity',
                    'PPV', 'NPV', 'Kappa', 'MCC']
    print(f"  {'Chỉ số':<14} {'Giá trị':>10}")
    print(f"  {'-'*26}")
    for col in display_cols:
        print(f"  {col:<14} {m[col]:>10.4f}")
    print(f"  {'-'*26}")
    print(f"  Threshold : {m['Threshold']:.4f}")
    print(f"  TP={m['TP']}  FP={m['FP']}  TN={m['TN']}  FN={m['FN']}")


def print_fold_summary_table(fold_metrics_list):
    cols = METRICS_COLS
    col_w = 13
    sep   = '=' * (8 + col_w * len(cols))

    header = f"  {'Fold':<8}" + "".join(f"{c:>{col_w}}" for c in cols)
    print(f"\n{sep}")
    print("  BẢNG TỔNG HỢP TỪNG FOLD")
    print(sep)
    print(header)
    print(f"  {'-'*(6 + col_w * len(cols))}")

    for i, m in enumerate(fold_metrics_list):
        row = f"  {f'Fold {i+1}':<8}" + "".join(f"{m[c]:>{col_w}.4f}" for c in cols)
        print(row)

    print(f"  {'-'*(6 + col_w * len(cols))}")

    means = {c: float(np.mean([m[c] for m in fold_metrics_list])) for c in cols}
    stds  = {c: float(np.std( [m[c] for m in fold_metrics_list], ddof=1)) for c in cols}

    print(f"  {'Mean':<8}" + "".join(f"{means[c]:>{col_w}.4f}" for c in cols))
    print(f"  {'±Std':<8}" + "".join(f"{stds[c]:>{col_w}.4f}"  for c in cols))
    print(sep)

    auroc_mean = means['AUROC']
    auroc_std  = stds['AUROC']
    ci_lo = auroc_mean - 1.96 * auroc_std
    ci_hi = auroc_mean + 1.96 * auroc_std
    print(f"\n  AUROC: {auroc_mean:.4f} ± {auroc_std:.4f}  "
          f"(95% CI: {ci_lo:.4f} – {ci_hi:.4f})")

    return means, stds


if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nThiết bị sử dụng: {device}")

    n_total     = len(target)
    n_neg_total = int(np.sum(target == 0))
    n_pos_total = int(np.sum(target == 1))
    print(f"Âm tính: {n_neg_total} | Dương tính: {n_pos_total}")

    preprocessor = TabularPreprocessor()
    preprocessor.fit(df, categorical_cols, numerical_cols)
    df_processed = preprocessor.transform(df, categorical_cols, numerical_cols)

    cat_data = (df_processed[categorical_cols].values if categorical_cols
                else np.empty((len(df_processed), 0), dtype=int))
    num_data = df_processed[numerical_cols].values if numerical_cols else None

    N_FOLDS = 5
    kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    fold_metrics_list = []
    fold_thresholds   = []
    all_val_preds     = np.zeros(len(df), dtype=np.float64)

    for fold, (train_idx, val_idx) in enumerate(kf.split(cat_data, target)):
        print(f"\n{'─'*60}")
        print(f"  FOLD {fold + 1} / {N_FOLDS}")
        print(f"{'─'*60}")

        cat_train, cat_val = cat_data[train_idx], cat_data[val_idx]
        num_train = num_data[train_idx] if num_data is not None else None
        num_val   = num_data[val_idx]   if num_data is not None else None
        y_train, y_val = target[train_idx], target[val_idx]

        n_pos_f  = int(np.sum(y_train == 1))
        n_neg_f  = int(np.sum(y_train == 0))
        n_train_f = len(y_train)
        w_pos_f  = n_train_f / (2.0 * n_pos_f)
        w_neg_f  = n_train_f / (2.0 * n_neg_f)
        print(f"  Train: {n_train_f} | Pos: {n_pos_f} | Neg: {n_neg_f}")
        print(f"  w_pos: {w_pos_f:.4f} | w_neg: {w_neg_f:.4f}")

        train_dataset = DengueDataset(cat_train, num_train, y_train)
        val_dataset   = DengueDataset(cat_val,   num_val,   y_val)

        model, _ = train_saint_model(
            train_dataset, val_dataset,
            preprocessor.categorical_dims,
            len(numerical_cols) if numerical_cols else 0,
            epochs=50, batch_size=64, lr=0.001, device=device,
            weight_pos=w_pos_f, weight_neg=w_neg_f
        )

        model.eval()
        fold_probs = []
        val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
        with torch.no_grad():
            for cat_batch, num_batch, _ in val_loader:
                cat_batch = cat_batch.to(device)
                num_batch = num_batch.to(device) if num_batch.numel() > 0 else None
                outputs   = model(cat_batch, num_batch).squeeze()
                if outputs.dim() == 0:
                    outputs = outputs.unsqueeze(0)
                fold_probs.extend(torch.sigmoid(outputs).cpu().numpy().tolist())

        all_val_preds[val_idx] = fold_probs

        # MCC-based threshold tuning
        best_thresh, best_mcc_val = find_best_threshold_mcc(y_val, fold_probs)
        fold_thresholds.append(best_thresh)
        print(f"\n  Optimal threshold (MCC): {best_thresh:.4f}  "
              f"(MCC = {best_mcc_val:.4f})")

        fold_m = compute_metrics(y_val, fold_probs, threshold=best_thresh)
        fold_metrics_list.append(fold_m)
        print_metrics_table(fold_m, title=f'Kết quả Fold {fold + 1}')

    means, stds = print_fold_summary_table(fold_metrics_list)

    mean_threshold = float(np.mean(fold_thresholds))
    print(f"\n  Fold thresholds : {[f'{t:.4f}' for t in fold_thresholds]}")
    print(f"  Mean threshold  : {mean_threshold:.4f}")

    oof_metrics = compute_metrics(target, all_val_preds, threshold=mean_threshold)
    print_metrics_table(
        oof_metrics,
        title=f'KẾT QUẢ CUỐI CÙNG (OOF — mean threshold = {mean_threshold:.4f})'
    )

    print(f"\n  Ma trận nhầm lẫn (threshold = {mean_threshold:.4f}):")
    print(f"  {'':22} {'Dự đoán Âm':>14} {'Dự đoán Dương':>14}")
    print(f"  {'Thực tế  Âm tính':22} {oof_metrics['TN']:>14} {oof_metrics['FP']:>14}")
    print(f"  {'Thực tế  Dương tính':22} {oof_metrics['FN']:>14} {oof_metrics['TP']:>14}")

    rows_detail = [
        {'Fold': f'Fold {i+1}', **{c: round(m[c], 4) for c in METRICS_COLS}}
        for i, m in enumerate(fold_metrics_list)
    ]
    pd.DataFrame(rows_detail).to_csv('fold_metrics.csv', index=False)
    print(f"\n  Đã lưu: fold_metrics.csv")

    rows_summary = [
        {'Split': f'Fold {i+1}', **{c: round(m[c], 4) for c in METRICS_COLS}}
        for i, m in enumerate(fold_metrics_list)
    ]
    rows_summary.append({
        'Split': 'Mean',
        **{c: round(means[c], 4) for c in METRICS_COLS}
    })
    rows_summary.append({
        'Split': '±Std (sample)',
        **{c: round(stds[c], 4) for c in METRICS_COLS}
    })
    rows_summary.append({
        'Split': 'OOF (mean_threshold)',
        **{c: round(oof_metrics[c], 4) for c in METRICS_COLS}
    })

    df_summary = pd.DataFrame(rows_summary).set_index('Split')
    df_summary.to_csv('fold_metrics_summary.csv')
    print(f"  Đã lưu: fold_metrics_summary.csv")

    print(f"\n{'='*60}")
    print("  HOÀN TẤT!")
    print(f"  Phương pháp  : Stratified {N_FOLDS}-Fold CV")
    print(f"  Loss         : Class-weighted BCE (không SMOTE)")
    print(f"  Threshold    : MCC-based per fold mean = {mean_threshold:.4f}")
    print(f"  Std (ddof=1) : Sample std — phù hợp N_FOLDS nhỏ")
    print(f"{'='*60}")

    print("\n=== DataFrame Kết Quả ===")
    print(df_summary.to_string())

Tổng số mẫu: 931
Kích thước dữ liệu: (931, 9)
Phân bố nhãn: Âm tính=300 | Dương tính=631
Tỷ lệ dương tính: 67.8%

Đặc trưng phân loại (3): ['Sex', 'Differential Count', 'RBC PANEL']
Đặc trưng số học    (5): ['Haemoglobin', 'Age', 'WBC Count', 'Platelet Count', 'PDW']
Tổng đặc trưng: 8

Thiết bị sử dụng: cuda
Âm tính: 300 | Dương tính: 631

────────────────────────────────────────────────────────────
  FOLD 1 / 5
────────────────────────────────────────────────────────────
  Train: 744 | Pos: 504 | Neg: 240
  w_pos: 0.7381 | w_neg: 1.5500
  Epoch   0: Train Loss=0.6352 | Val Loss=0.5001 | Val AUC=1.0000
  Epoch  10: Train Loss=0.0015 | Val Loss=0.0004 | Val AUC=1.0000
  Epoch  20: Train Loss=0.0007 | Val Loss=0.0002 | Val AUC=1.0000
  Epoch  30: Train Loss=0.0005 | Val Loss=0.0000 | Val AUC=1.0000
  Epoch  40: Train Loss=0.0002 | Val Loss=0.0000 | Val AUC=1.0000

  → Optimal threshold (MCC): 0.3900  (MCC = 0.9879)

  Kết quả Fold 1
  Chỉ số            Giá trị
  -------------------------